In [0]:
%sql
CREATE VOLUME IF NOT EXISTS retail_lakehouse.bronze.raw_files;

In [0]:
display(dbutils.fs.ls("/Volumes/retail_lakehouse/bronze/raw_files/"))

In [0]:
source_path = "/Volumes/retail_lakehouse/bronze/raw_files/sales_2026_05_01.csv"

sales_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(source_path)
)

display(sales_df)

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col


bronze_sales_df = (
    sales_df
    .withColumn("ingestion_time", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
    .withColumn("pipeline_id", lit("sales_pipeline"))
)

bronze_sales_df.write \
    .mode("append") \
    .format("delta") \
    .saveAsTable("retail_lakehouse.bronze.sales")

In [0]:
%sql
SELECT * FROM retail_lakehouse.bronze.sales;